In [4]:
from datetime import date

import pandas as pd
import numpy as np
import holidays
import yaml

In [ ]:
dim_tiempo = pd.DataFrame({
    "fecha": pd.fecha_range(start='09/19/2023', end='31/08/2024', freq='D')
})
dim_tiempo.head()

,fecha
0,2023-09-19
1,2023-09-20
2,2023-09-21
3,2023-09-22
4,2023-09-23


In [31]:
dim_tiempo["año"] = dim_tiempo["fecha"].dt.year
dim_tiempo["mes"] = dim_tiempo["fecha"].dt.month
dim_tiempo["dia"] = dim_tiempo["fecha"].dt.day
dim_tiempo["dia_semana"] = dim_tiempo["fecha"].dt.weekday
dim_tiempo["fin_de_semana"] = np.where(dim_tiempo["dia_semana"].isin([5, 6]), 1, 0)
dim_tiempo.head()

,fecha,año,mes,dia,dia_semana,fin_de_semana,dia_mes
0,2023-09-19,2023,9,19,1,0,30
1,2023-09-20,2023,9,20,2,0,30
2,2023-09-21,2023,9,21,3,0,30
3,2023-09-22,2023,9,22,4,0,30
4,2023-09-23,2023,9,23,5,1,30


In [34]:
from sqlalchemy import create_engine

with open('../config_fill.yml', 'r') as f:
    config = yaml.safe_load(f)
    config_co = config['mensajeria_bd']
    config_etl = config['etl_mensajeria']

# Construct the database URL
url_co = (f"{config_co['drivername']}://{config_co['user']}:{config_co['password']}@{config_co['host']}:"
          f"{config_co['port']}/{config_co['dbname']}")
url_etl = (f"{config_etl['drivername']}://{config_etl['user']}:{config_etl['password']}@{config_etl['host']}:"
           f"{config_etl['port']}/{config_etl['dbname']}")
# Create the SQLAlchemy Engine
mensajeria_bd = create_engine(url_co)
etl_mensajeria = create_engine(url_etl)

In [35]:
dim_tiempo.to_sql('dim_tiempo', etl_mensajeria, if_exists='replace',index_label='key_dim_fecha')

348